In [1]:
# ==============================================================================
# Model 17: Siamese BiGRU + Tier 1 + Tier 2 Improvements
# Base: Model 16 (all Tier 1 changes)
# New changes vs Model 16:
#   T2-A. Label smoothing on KL target  (prevents overfitting noisy annotator dist)
#   T2-B. Contrastive sense-pair loss   (pushes same-story different-meaning scores apart)
#   T2-C. Fixed cross-attention         (separate MAX_LEN_MEANING=40, proper padding mask)
#         Architecture: BiGRU returns sequences -> CrossAttention -> pool
# ==============================================================================
!wget -q https://nlp.stanford.edu/data/glove.6B.zip
!unzip -q -o glove.6B.zip
!pip install -q optuna

import json
import os
import re
import random
from contextlib import nullcontext

import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from torch.utils.data import DataLoader, Dataset, Sampler

# ------------------------------------------------------------------------------
# 1. Setup & Config
# ------------------------------------------------------------------------------
SEED           = 42
ENSEMBLE_SEEDS = [42, 123, 777, 2024, 99]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

TRAIN_JSON  = "/kaggle/input/datasets/satyagudu/inlp-proj/train.json"
DEV_JSON    = "/kaggle/input/datasets/satyagudu/inlp-proj/dev.json"
TEST_JSON   = "/kaggle/input/datasets/satyagudu/inlp-proj/test.json"
GLOVE_PATH  = "/kaggle/working/glove.6B.300d.txt"

MAX_WORDS        = 12000
MAX_LEN_STORY    = 150    # story+precontext+ending padded to this
MAX_LEN_MEANING  = 40     # T2-C: meaning is short, separate shorter length
EMBED_DIM        = 300

# T2-A: label smoothing strength (0.0 = off, 0.05 is mild)
LABEL_SMOOTH_ALPHA = 0.05

TUNE_EPOCHS  = 18
FINAL_EPOCHS = 36
PATIENCE     = 5
N_TRIALS     = 35

HOM_OPEN  = "<HOM>"
HOM_CLOSE = "</HOM>"

if not (os.path.exists(TRAIN_JSON) and os.path.exists(DEV_JSON) and os.path.exists(TEST_JSON)):
    print("WARNING: Please upload train.json, dev.json and test.json before running.")

# ------------------------------------------------------------------------------
# 2. Data Processing Helpers  (Tier 1 helpers unchanged)
# ------------------------------------------------------------------------------
def mark_homonym(sentence, homonym):
    if not homonym:
        return sentence
    pattern = r'\b' + re.escape(homonym.lower()) + r'\b'
    return re.sub(pattern,
                  f'{HOM_OPEN} {homonym.lower()} {HOM_CLOSE}',
                  sentence, flags=re.IGNORECASE)

def build_story(item):
    hom  = item.get("homonym", "")
    sent = mark_homonym(item.get("sentence", ""), hom)
    return " ".join([item.get("precontext", ""), sent, item.get("ending", "")])

def build_meaning(item):
    defn = item.get("judged_meaning", "")
    ex   = item.get("example_sentence", "")
    return f"{defn} {ex}".strip()

def build_setup_key(item):
    return "|||".join([
        item.get("precontext", ""),
        item.get("sentence",  ""),
        item.get("ending",    ""),
    ])

def build_sense_key(item):
    """T2-B: Key that identifies the (story_setup, meaning) pair.
    Two items sharing the same setup_key but different sense_keys are
    contrastive pairs: same story, different word-sense rating."""
    return "|||".join([
        item.get("precontext",    ""),
        item.get("sentence",      ""),
        item.get("ending",        ""),
        item.get("judged_meaning",""),
    ])

def normalize_label_distribution(choices):
    counts = np.zeros(5, dtype=np.float32)
    for c in choices:
        c = int(c)
        if 1 <= c <= 5:
            counts[c - 1] += 1.0
    total = counts.sum()
    return counts / total if total > 0 else np.ones(5, dtype=np.float32) / 5.0

def smooth_distribution(dist, alpha=LABEL_SMOOTH_ALPHA):
    """T2-A: Blend the raw annotator distribution with a uniform prior.
    This prevents the model from over-fitting noise in a 5-annotator sample.
    alpha=0.05 means 5% uniform, 95% raw distribution."""
    return (1.0 - alpha) * dist + alpha / dist.shape[-1]

def get_amp_autocast(device):
    if device.type != "cuda":
        return nullcontext()
    if hasattr(torch, "autocast"):
        return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True)
    return torch.cuda.amp.autocast(dtype=torch.float16, enabled=True)

def get_grad_scaler(device):
    amp_enabled = device.type == "cuda"
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler(device.type, enabled=amp_enabled)
    return torch.cuda.amp.GradScaler(enabled=amp_enabled)

# ------------------------------------------------------------------------------
# 3. Load & Prepare Train+Dev Data
# ------------------------------------------------------------------------------
with open(TRAIN_JSON, encoding="utf8") as f:
    raw_train = json.load(f)
with open(DEV_JSON, encoding="utf8") as f:
    raw_dev = json.load(f)

train_items  = list(raw_train.values()) + list(raw_dev.values())
print(f"Loaded train={len(raw_train)} + dev={len(raw_dev)} => merged={len(train_items)}")
stories      = [build_story(item)   for item in train_items]
meanings     = [build_meaning(item) for item in train_items]

labels      = np.array([i["average"] for i in train_items], dtype=np.float32)
stdevs      = np.array([i["stdev"]   for i in train_items], dtype=np.float32)
label_dists = np.array(
    [normalize_label_distribution(i.get("choices", [])) for i in train_items],
    dtype=np.float32
)

# T2-A: apply label smoothing to all training distributions
label_dists_smooth = smooth_distribution(label_dists)

setup_keys   = [build_setup_key(i) for i in train_items]
unique_setup = {k: idx for idx, k in enumerate(sorted(set(setup_keys)))}
setup_ids    = np.array([unique_setup[k] for k in setup_keys], dtype=np.int64)

# T2-B: sense-group IDs (each unique story×meaning pair gets an ID)
sense_keys   = [build_sense_key(i) for i in train_items]
unique_sense = {k: idx for idx, k in enumerate(sorted(set(sense_keys)))}
sense_ids    = np.array([unique_sense[k] for k in sense_keys], dtype=np.int64)

homonyms    = [i.get("homonym", "<UNK>") for i in train_items]
unique_hom  = {h: idx for idx, h in enumerate(sorted(set(homonyms)), start=1)}
hom_ids     = np.array([unique_hom.get(h, 0) for h in homonyms], dtype=np.int64)

nons_count  = np.array(
    [sum(1 for v in i.get("nonsensical", []) if v) for i in train_items],
    dtype=np.float32
)
ending_flag = np.array(
    [1.0 if len(i.get("ending", "").strip()) > 0 else 0.0 for i in train_items],
    dtype=np.float32
)
meta_feats  = np.stack(
    [np.clip(nons_count / 5.0, 0.0, 1.0), ending_flag], axis=1
).astype(np.float32)

# Tokenizer: filters keep < > / so <HOM> </HOM> survive
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>",
                      filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(stories + meanings)
for tok in [HOM_OPEN, HOM_CLOSE]:
    if tok not in tokenizer.word_index:
        tokenizer.word_index[tok]  = len(tokenizer.word_index) + 1
        tokenizer.index_word[tokenizer.word_index[tok]] = tok

# T2-C: stories pad to MAX_LEN_STORY, meanings pad to MAX_LEN_MEANING
X_story = pad_sequences(
    tokenizer.texts_to_sequences(stories),
    maxlen=MAX_LEN_STORY, padding="post"
)
X_mean = pad_sequences(
    tokenizer.texts_to_sequences(meanings),
    maxlen=MAX_LEN_MEANING, padding="post"
)

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(gss.split(X_story, labels, groups=setup_ids))

Xs_train, Xs_val         = X_story[train_idx],         X_story[val_idx]
Xm_train, Xm_val         = X_mean[train_idx],          X_mean[val_idx]
y_train,  y_val          = labels[train_idx],           labels[val_idx]
dist_train, dist_val     = label_dists_smooth[train_idx], label_dists_smooth[val_idx]
s_train,  s_val          = stdevs[train_idx],           stdevs[val_idx]
setup_train, setup_val   = setup_ids[train_idx],        setup_ids[val_idx]
sense_train, sense_val   = sense_ids[train_idx],        sense_ids[val_idx]
hom_train,  hom_val      = hom_ids[train_idx],          hom_ids[val_idx]
meta_train, meta_val     = meta_feats[train_idx],       meta_feats[val_idx]

print("Loading GloVe 300d embeddings...")
embeddings_index = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        vals = line.split()
        embeddings_index[vals[0]] = np.asarray(vals[1:], dtype="float32")
print(f"Loaded {len(embeddings_index):,} GloVe vectors.")

word_index       = tokenizer.word_index
num_words        = min(MAX_WORDS, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, EMBED_DIM), dtype=np.float32)
for word, idx in word_index.items():
    if idx < num_words and word in embeddings_index:
        embedding_matrix[idx] = embeddings_index[word]
for tok in [HOM_OPEN, HOM_CLOSE]:
    idx = tokenizer.word_index.get(tok)
    if idx is not None and idx < num_words:
        embedding_matrix[idx] = np.random.randn(EMBED_DIM).astype(np.float32) * 0.01

print(f"Embedding matrix: {embedding_matrix.shape}")

# ------------------------------------------------------------------------------
# 4. Model Architecture
# ------------------------------------------------------------------------------
class BiGRUSequenceEncoder(nn.Module):
    """T2-C: Returns the full token sequence [B, L, 2h] instead of pooled vector.
    Cross-attention needs the token-level representations."""
    def __init__(self, num_words, embed_dim, embedding_matrix,
                 rnn_units, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(num_words, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(embedding_matrix, dtype=torch.float32)
        )
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=rnn_units,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        emb      = self.dropout(self.embedding(x))  # [B, L, embed_dim]
        out, _   = self.gru(emb)                    # [B, L, 2*rnn_units]
        return out


class CrossAttentionPooling(nn.Module):
    """T2-C: Bidirectional cross-attention between two sequences.

    Fixes from Model 11:
      - key_padding_mask: ignores padding tokens (token_id == 0) in attention.
        Without this, softmax spreads weight across 100+ PAD tokens -> noise.
      - Separate MAX_LEN_MEANING=40 so the meaning sequence is compact and
        the key matrix is small enough to be informative.
    """
    def __init__(self, dim):
        super().__init__()
        self.scale = dim ** -0.5

    def _attend(self, query_seq, key_seq, key_pad_mask):
        """query_seq: [B, Lq, D], key_seq: [B, Lk, D], key_pad_mask: [B, Lk] bool.
        Returns pooled attended vector [B, 2D] via mean+max pooling."""
        # Attention scores: [B, Lq, Lk]
        scores = torch.bmm(query_seq, key_seq.transpose(1, 2)) * self.scale
        # Mask padding positions in the key to -inf before softmax
        if key_pad_mask is not None:
            scores = scores.masked_fill(key_pad_mask.unsqueeze(1), float('-inf'))
        weights   = torch.softmax(scores, dim=-1)
        # Guard against all-inf rows (fully padded sequences) -> NaN after softmax
        weights   = torch.nan_to_num(weights, nan=0.0)
        attended  = torch.bmm(weights, key_seq)          # [B, Lq, D]
        mean_pool = attended.mean(dim=1)                 # [B, D]
        max_pool  = attended.max(dim=1).values           # [B, D]
        return torch.cat([mean_pool, max_pool], dim=1)   # [B, 2D]

    def forward(self, story_seq, meaning_seq, story_pad_mask, meaning_pad_mask):
        """Returns (sv_attended [B,2D], mv_attended [B,2D])."""
        sv = self._attend(story_seq,   meaning_seq, meaning_pad_mask)  # story attends meaning
        mv = self._attend(meaning_seq, story_seq,   story_pad_mask)    # meaning attends story
        return sv, mv


class Model16Network(nn.Module):
    """Siamese BiGRU with cross-attention (T2-C), homonym embedding, meta features."""
    def __init__(self, num_words, embed_dim, embedding_matrix,
                 rnn_units, num_layers, dropout,
                 dense_units, hom_vocab_size, hom_emb_dim, meta_dim):
        super().__init__()
        self.shared_encoder = BiGRUSequenceEncoder(
            num_words, embed_dim, embedding_matrix, rnn_units, num_layers, dropout
        )
        self.cross_attn = CrossAttentionPooling(dim=rnn_units * 2)
        self.hom_emb    = nn.Embedding(hom_vocab_size + 1, hom_emb_dim)

        # sv and mv are each [B, 2 * (2*rnn_units)] = [B, 4h] after mean+max pooling
        # comparison vector: [sv; mv; |sv-mv|; sv*mv; cos; hom_vec; meta]
        h = rnn_units * 2           # hidden dim of one direction's output per token
        pooled_dim   = h * 2        # mean_pool + max_pool = 2h per branch
        combined_dim = (pooled_dim * 4) + 1 + hom_emb_dim + meta_dim

        self.head = nn.Sequential(
            nn.Linear(combined_dim, dense_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dense_units, 5),
        )

    def forward(self, story, meaning, hom_ids, meta):
        # Build padding masks (True = position is padding, should be ignored)
        story_pad_mask   = (story   == 0)   # [B, MAX_LEN_STORY]
        meaning_pad_mask = (meaning == 0)   # [B, MAX_LEN_MEANING]

        # Encode both sequences with shared weights
        story_seq   = self.shared_encoder(story)    # [B, MAX_LEN_STORY,   2h]
        meaning_seq = self.shared_encoder(meaning)  # [B, MAX_LEN_MEANING, 2h]

        # Cross-attend: story looks at meaning, meaning looks at story
        sv, mv = self.cross_attn(
            story_seq, meaning_seq, story_pad_mask, meaning_pad_mask
        )  # sv: [B, 4h], mv: [B, 4h]

        abs_diff = torch.abs(sv - mv)
        prod     = sv * mv
        cosine   = F.cosine_similarity(sv, mv, dim=1, eps=1e-8).unsqueeze(1)
        hom_vec  = self.hom_emb(hom_ids)
        combined = torch.cat([sv, mv, abs_diff, prod, cosine, hom_vec, meta], dim=1)
        return self.head(combined)

# ------------------------------------------------------------------------------
# 5. Dataset — extended to carry sense_ids for contrastive loss
# ------------------------------------------------------------------------------
class AmbiStoryDataset(Dataset):
    def __init__(self, X_story, X_mean, labels, dists, stdevs,
                 setup_ids, sense_ids, hom_ids, meta_feats):
        self.X_story   = torch.tensor(X_story,    dtype=torch.long)
        self.X_mean    = torch.tensor(X_mean,     dtype=torch.long)
        self.labels    = torch.tensor(labels,     dtype=torch.float32)
        self.dists     = torch.tensor(dists,      dtype=torch.float32)
        self.stdevs    = torch.tensor(stdevs,     dtype=torch.float32)
        self.setup_ids = torch.tensor(setup_ids,  dtype=torch.long)
        self.sense_ids = torch.tensor(sense_ids,  dtype=torch.long)
        self.hom_ids   = torch.tensor(hom_ids,    dtype=torch.long)
        self.meta      = torch.tensor(meta_feats, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.X_story[idx], self.X_mean[idx],
            self.labels[idx],  self.dists[idx],
            self.stdevs[idx],  self.setup_ids[idx],
            self.sense_ids[idx], self.hom_ids[idx],
            self.meta[idx],
        )


class SetupBatchSampler(Sampler):
    """Groups samples by setup so ranking and contrastive losses see related pairs."""
    def __init__(self, setup_ids, setups_per_batch=8, shuffle=True):
        self.shuffle          = shuffle
        self.setups_per_batch = setups_per_batch
        self.indices_by_setup = {}
        for idx, sid in enumerate(setup_ids):
            self.indices_by_setup.setdefault(int(sid), []).append(idx)
        self.setup_keys = list(self.indices_by_setup.keys())

    def __iter__(self):
        keys = self.setup_keys.copy()
        if self.shuffle:
            random.shuffle(keys)
        for start in range(0, len(keys), self.setups_per_batch):
            selected = keys[start: start + self.setups_per_batch]
            batch = []
            for sid in selected:
                inds = self.indices_by_setup[sid]
                if self.shuffle:
                    random.shuffle(inds)
                batch.extend(inds)
            if batch:
                yield batch

    def __len__(self):
        return (len(self.setup_keys) + self.setups_per_batch - 1) // self.setups_per_batch

# ------------------------------------------------------------------------------
# 6. Losses
# ------------------------------------------------------------------------------
def get_expected_scores(logits):
    probs   = torch.softmax(logits, dim=-1)
    weights = torch.arange(1, 6, device=logits.device, dtype=torch.float32)
    return (probs * weights).sum(dim=-1)


def pairwise_setup_ranking_loss(preds, targets, setup_ids, margin=0.2):
    """Margin ranking loss across all pairs within each story setup."""
    total_loss = preds.new_tensor(0.0)
    groups     = 0
    for sid in torch.unique(setup_ids):
        mask = setup_ids == sid
        p, y = preds[mask], targets[mask]
        if p.numel() < 2:
            continue
        y_diff   = y.unsqueeze(1) - y.unsqueeze(0)
        p_diff   = p.unsqueeze(1) - p.unsqueeze(0)
        valid    = y_diff.abs() > 1e-6
        if valid.sum() == 0:
            continue
        loss_mat = torch.relu(margin - y_diff.sign() * p_diff)
        total_loss = total_loss + (loss_mat * valid).sum() / valid.sum()
        groups += 1
    return total_loss / groups if groups > 0 else preds.new_tensor(0.0)


def contrastive_sense_loss(preds, targets, setup_ids, sense_ids, margin=0.3):
    """T2-B: Contrastive loss between different word-sense ratings.

    For each pair of items (i, j) that share the same story setup but have
    DIFFERENT sense_ids (i.e. different judged_meaning for the same story),
    we apply a margin ranking loss.

    Intuition: if annotators rated sense-A as 4.0 and sense-B as 1.5 for the
    same story, the model should push predicted(A) > predicted(B) by at least
    `margin`.  This is the contrastive signal between meanings that the plain
    pairwise ranking loss also captures, but here we weight only cross-sense
    pairs — same-sense pairs are skipped (they differ in ending, not meaning).
    """
    total_loss = preds.new_tensor(0.0)
    groups     = 0
    for sid in torch.unique(setup_ids):
        mask = setup_ids == sid
        p, y, sn = preds[mask], targets[mask], sense_ids[mask]
        if p.numel() < 2:
            continue
        # cross-sense mask: i and j belong to different senses for this setup
        cross_sense = sn.unsqueeze(1) != sn.unsqueeze(0)
        y_diff      = y.unsqueeze(1) - y.unsqueeze(0)
        p_diff      = p.unsqueeze(1) - p.unsqueeze(0)
        # Only penalise where labels differ AND the pair crosses senses
        valid = cross_sense & (y_diff.abs() > 1e-6)
        if valid.sum() == 0:
            continue
        loss_mat = torch.relu(margin - y_diff.sign() * p_diff)
        total_loss = total_loss + (loss_mat * valid).sum() / valid.sum()
        groups += 1
    return total_loss / groups if groups > 0 else preds.new_tensor(0.0)


# ------------------------------------------------------------------------------
# 7. Training Loop
# ------------------------------------------------------------------------------
def create_loaders(batch_size_setups):
    workers      = 2 if device.type == "cuda" else 0
    train_ds     = AmbiStoryDataset(
        Xs_train, Xm_train, y_train, dist_train,
        s_train, setup_train, sense_train, hom_train, meta_train
    )
    val_ds       = AmbiStoryDataset(
        Xs_val, Xm_val, y_val, dist_val,
        s_val, setup_val, sense_val, hom_val, meta_val
    )
    train_sampler = SetupBatchSampler(
        setup_train, setups_per_batch=batch_size_setups, shuffle=True
    )
    train_loader  = DataLoader(
        train_ds, batch_sampler=train_sampler,
        pin_memory=(device.type == "cuda"), num_workers=workers,
    )
    val_loader    = DataLoader(
        val_ds, batch_size=64, shuffle=False,
        pin_memory=(device.type == "cuda"), num_workers=workers,
    )
    return train_loader, val_loader


def train_one_epoch(model, loader, optimizer, scaler, huber_loss_fn,
                    rank_lambda, rank_margin, huber_lambda,
                    contrast_lambda, contrast_margin):
    model.train()
    total_loss = 0.0
    for story, meaning, label, dist, _stdev, sid, sense, hom, meta in loader:
        story, meaning, label = story.to(device), meaning.to(device), label.to(device)
        dist  = dist.to(device)
        sid   = sid.to(device)
        sense = sense.to(device)
        hom   = hom.to(device)
        meta  = meta.to(device)

        optimizer.zero_grad(set_to_none=True)
        with get_amp_autocast(device):
            logits          = model(story, meaning, hom, meta)
            expected_scores = get_expected_scores(logits)

            # T2-A: dist already label-smoothed; KL trains against smooth target
            loss_kl       = F.kl_div(F.log_softmax(logits, dim=-1), dist, reduction="batchmean")
            loss_rank     = pairwise_setup_ranking_loss(expected_scores, label, sid, margin=rank_margin)
            loss_huber    = huber_loss_fn(expected_scores, label)
            # T2-B: contrastive loss between different senses of same story
            loss_contrast = contrastive_sense_loss(expected_scores, label, sid, sense, margin=contrast_margin)

            loss = (
                loss_kl
                + rank_lambda     * loss_rank
                + huber_lambda    * loss_huber
                + contrast_lambda * loss_contrast
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def predict(model, Xs, Xm, hom_ids_arr, meta_arr, sense_arr=None):
    model.eval()
    n         = len(Xs)
    dummy_y   = np.zeros(n, dtype=np.float32)
    dummy_d   = np.zeros((n, 5), dtype=np.float32)
    dummy_s   = np.ones(n, dtype=np.float32)
    dummy_sid = np.arange(n, dtype=np.int64)
    dummy_sn  = np.zeros(n, dtype=np.int64) if sense_arr is None else sense_arr

    ds     = AmbiStoryDataset(
        Xs, Xm, dummy_y, dummy_d, dummy_s,
        dummy_sid, dummy_sn, hom_ids_arr, meta_arr
    )
    loader = DataLoader(ds, batch_size=64, shuffle=False,
                        pin_memory=(device.type == "cuda"))
    all_preds = []
    for story, meaning, _l, _d, _s, _sid, _sn, hom, meta in loader:
        story, meaning = story.to(device), meaning.to(device)
        hom,   meta    = hom.to(device),   meta.to(device)
        logits          = model(story, meaning, hom, meta)
        expected_scores = get_expected_scores(logits)
        all_preds.append(expected_scores.cpu().numpy())
    return np.clip(np.concatenate(all_preds), 1, 5)


def objective_score(y_true, y_pred, s_true):
    sp = spearmanr(y_true, y_pred).correlation
    if np.isnan(sp):
        sp = -1.0
    acc = np.mean(np.abs(y_pred - y_true) <= np.maximum(s_true, 1.0))
    mae = np.mean(np.abs(y_pred - y_true))
    return float(sp + 0.25 * acc - 0.05 * mae), float(sp), float(acc), float(mae)


def run_training(params, epochs, patience, seed=SEED, trial=None):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = Model16Network(
        num_words=num_words,
        embed_dim=EMBED_DIM,
        embedding_matrix=embedding_matrix,
        rnn_units=params["rnn_units"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        dense_units=params["dense_units"],
        hom_vocab_size=len(unique_hom),
        hom_emb_dim=params["hom_emb_dim"],
        meta_dim=meta_train.shape[1],
    ).to(device)

    optimizer     = optim.Adam(model.parameters(),
                               lr=params["lr"], weight_decay=params["weight_decay"])
    scheduler     = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=2, factor=0.5
    )
    scaler        = get_grad_scaler(device)
    huber_loss_fn = nn.HuberLoss(reduction="mean", delta=1.0)
    train_loader, val_loader = create_loaders(params["setup_batch"])

    best_score   = -1e9
    best_weights = None
    patience_ctr = 0

    for epoch in range(1, epochs + 1):
        _ = train_one_epoch(
            model, train_loader, optimizer, scaler, huber_loss_fn,
            rank_lambda=params["rank_lambda"],
            rank_margin=params["rank_margin"],
            huber_lambda=params["huber_lambda"],
            contrast_lambda=params["contrast_lambda"],
            contrast_margin=params["contrast_margin"],
        )
        val_preds           = predict(model, Xs_val, Xm_val, hom_val, meta_val, sense_val)
        score, sp, acc, mae = objective_score(y_val, val_preds, s_val)
        scheduler.step(score)

        if trial is not None:
            trial.report(score, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if score > best_score:
            best_score   = score
            best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    if best_weights is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_weights.items()})

    final_preds         = predict(model, Xs_val, Xm_val, hom_val, meta_val, sense_val)
    score, sp, acc, mae = objective_score(y_val, final_preds, s_val)
    return model, score, sp, acc, mae

# ------------------------------------------------------------------------------
# 8. Optuna Hyperparameter Search
# ------------------------------------------------------------------------------
def objective(trial):
    params = {
        "rnn_units":       trial.suggest_categorical("rnn_units",       [64, 96, 128]),
        "num_layers":      trial.suggest_categorical("num_layers",      [1, 2]),
        "dropout":         trial.suggest_float("dropout",               0.15, 0.45),
        "dense_units":     trial.suggest_categorical("dense_units",     [96, 128, 192, 256]),
        "hom_emb_dim":     trial.suggest_categorical("hom_emb_dim",     [8, 16, 24]),
        "lr":              trial.suggest_float("lr",                    1e-4, 2e-3, log=True),
        "weight_decay":    trial.suggest_float("weight_decay",          1e-6, 5e-4, log=True),
        "setup_batch":     trial.suggest_categorical("setup_batch",     [6, 8, 10, 12]),
        "rank_lambda":     trial.suggest_float("rank_lambda",           0.1, 0.8),
        "rank_margin":     trial.suggest_float("rank_margin",           0.1, 0.5),
        "huber_lambda":    trial.suggest_float("huber_lambda",          0.1, 1.0),
        # T2-B: two new hyperparams for the contrastive loss
        "contrast_lambda": trial.suggest_float("contrast_lambda",       0.05, 0.6),
        "contrast_margin": trial.suggest_float("contrast_margin",       0.2, 1.0),
    }
    _, score, sp, acc, mae = run_training(
        params, epochs=TUNE_EPOCHS, patience=PATIENCE, seed=SEED, trial=trial
    )
    trial.set_user_attr("spearman", float(sp))
    trial.set_user_attr("acc_within_std", float(acc))
    trial.set_user_attr("mae", float(mae))
    return score



def optuna_metrics_callback(study, trial):
    sp = trial.user_attrs.get("spearman", float("nan"))
    acc = trial.user_attrs.get("acc_within_std", float("nan"))
    mae = trial.user_attrs.get("mae", float("nan"))
    print(
        f"[Trial {trial.number}] value={trial.value:.6f} | "
        f"Spearman={sp:.5f} | AccWithinStd={acc:.5f} | MAE={mae:.5f} | "
        f"Best={study.best_value:.6f}"
    )

def make_test_meta_and_hom(test_items):
    test_hom  = np.array(
        [unique_hom.get(i.get("homonym", "<UNK>"), 0) for i in test_items],
        dtype=np.int64
    )
    test_nons = np.array(
        [sum(1 for v in i.get("nonsensical", []) if v) for i in test_items],
        dtype=np.float32
    )
    test_end  = np.array(
        [1.0 if len(i.get("ending", "").strip()) > 0 else 0.0 for i in test_items],
        dtype=np.float32
    )
    test_meta = np.stack(
        [np.clip(test_nons / 5.0, 0.0, 1.0), test_end], axis=1
    ).astype(np.float32)
    return test_hom, test_meta

# ------------------------------------------------------------------------------
# 9. Main: Tune -> Multi-seed Ensemble -> Predict
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    if not (os.path.exists(TRAIN_JSON) and os.path.exists(DEV_JSON) and os.path.exists(TEST_JSON)):
        raise FileNotFoundError("Upload train.json, dev.json and test.json to /content/")

    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=5)
    study   = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True, callbacks=[optuna_metrics_callback])

    print("\nBest trial:")
    print(f"  Value (composite): {study.best_value:.5f}")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

    # Multi-seed ensemble
    print(f"\nTraining ensemble of {len(ENSEMBLE_SEEDS)} seeds with best params...")
    ensemble_models    = []
    ensemble_val_preds = []

    for seed in ENSEMBLE_SEEDS:
        print(f"  Seed {seed}...", end=" ")
        model, score, sp, acc, mae = run_training(
            study.best_params, epochs=FINAL_EPOCHS, patience=7, seed=seed
        )
        ensemble_models.append(model)
        val_p = predict(model, Xs_val, Xm_val, hom_val, meta_val, sense_val)
        ensemble_val_preds.append(val_p)
        print(f"Spearman={sp:.4f}  Acc={acc:.4f}  MAE={mae:.4f}")

    ensemble_val = np.mean(ensemble_val_preds, axis=0)
    _, ens_sp, ens_acc, ens_mae = objective_score(y_val, ensemble_val, s_val)

    print("\n=== Ensemble validation results ===")
    print(f"  Spearman r          : {ens_sp:.5f}")
    print(f"  Accuracy within std : {ens_acc:.5f}")
    print(f"  MAE                 : {ens_mae:.5f}")

    # Test predictions
    with open(TEST_JSON, encoding="utf8") as f:
        raw_test   = json.load(f)
    test_items = list(raw_test.values())

    test_stories  = [build_story(i)   for i in test_items]
    test_meanings = [build_meaning(i) for i in test_items]

    test_story_seq = pad_sequences(
        tokenizer.texts_to_sequences(test_stories),
        maxlen=MAX_LEN_STORY, padding="post"
    )
    test_mean_seq = pad_sequences(
        tokenizer.texts_to_sequences(test_meanings),
        maxlen=MAX_LEN_MEANING, padding="post"
    )
    test_hom, test_meta = make_test_meta_and_hom(test_items)

    test_preds_all = [
        predict(m, test_story_seq, test_mean_seq, test_hom, test_meta)
        for m in ensemble_models
    ]
    test_preds = np.mean(test_preds_all, axis=0)

    submission = {
        key: float(round(pred, 4))
        for key, pred in zip(raw_test.keys(), test_preds)
    }
    with open("/content/submission_model17_train_dev.json", "w") as f:
        json.dump(submission, f, indent=2)
    print("\nSubmission saved to /content/submission_model17_train_dev.json")
    print(f"Predictions — min: {test_preds.min():.3f}  max: {test_preds.max():.3f}  mean: {test_preds.mean():.3f}")


2026-04-08 05:55:14.591154: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775627714.758600      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775627714.808155      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775627715.260465      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775627715.260499      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775627715.260502      55 computation_placer.cc:177] computation placer alr

Using device: cuda
Loaded train=2280 + dev=588 => merged=2868
Loading GloVe 300d embeddings...


[I 2026-04-08 05:55:55,450] A new study created in memory with name: no-name-ce8c0778-a3b4-479a-9611-53f22138b8d7


Loaded 400,000 GloVe vectors.
Embedding matrix: (5875, 300)


  0%|          | 0/35 [00:00<?, ?it/s]

[I 2026-04-08 05:57:13,158] Trial 0 finished with value: 0.6969964851435353 and parameters: {'rnn_units': 96, 'num_layers': 1, 'dropout': 0.1967983561008608, 'dense_units': 128, 'hom_emb_dim': 16, 'lr': 0.00018891200276189413, 'weight_decay': 3.0955664602423687e-06, 'setup_batch': 10, 'rank_lambda': 0.30386039813862936, 'rank_margin': 0.34474115788895177, 'huber_lambda': 0.22554447458683766, 'contrast_lambda': 0.21067955669436994, 'contrast_margin': 0.4930894746349534}. Best is trial 0 with value: 0.6969964851435353.
[Trial 0] value=0.696996 | Spearman=0.54375 | AccWithinStd=0.76852 | MAE=0.77774 | Best=0.696996
[I 2026-04-08 05:58:30,875] Trial 1 finished with value: 0.6799320321763332 and parameters: {'rnn_units': 96, 'num_layers': 2, 'dropout': 0.1639351238159993, 'dense_units': 256, 'hom_emb_dim': 8, 'lr': 0.0001339906056150979, 'weight_decay': 7.02626320544305e-05, 'setup_batch': 10, 'rank_lambda': 0.7365242814551475, 'rank_margin': 0.20351199264000677, 'huber_lambda': 0.696270055

In [2]:
# Save submission to Kaggle working directory
import json
import os
import shutil

kaggle_out = "/kaggle/working/submission_model17_train_dev.json"
os.makedirs("/kaggle/working", exist_ok=True)

if "raw_test" in globals() and "test_preds" in globals():
    submission_kaggle = {
        key: float(round(pred, 4))
        for key, pred in zip(raw_test.keys(), test_preds)
    }
    with open(kaggle_out, "w", encoding="utf8") as f:
        json.dump(submission_kaggle, f, indent=2)
    print(f"Saved Kaggle submission to {kaggle_out}")
else:
    src = "/content/submission_model17_train_dev.json"
    if os.path.exists(src):
        shutil.copy2(src, kaggle_out)
        print(f"Copied {src} -> {kaggle_out}")
    else:
        raise FileNotFoundError(
            "No predictions found in memory and /content/submission_model17_train_dev.json is missing. Run Cell 1 first."
        )

Saved Kaggle submission to /kaggle/working/submission_model17_train_dev.json
